# Environment Preparation and Data Generation
Assume Image shape (32, 32, 3), batch size 2

In [2]:
import torch
import torch.nn as nn
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

In [3]:
# dummy image
batch_size = 2
channels = 3
image_size = 32
x = torch.randn(batch_size, channels, image_size, image_size)

print(f"Org image shape: {x.shape} -> [Batch, Channels, Height, Width]")

Org image shape: torch.Size([2, 3, 32, 32]) -> [Batch, Channels, Height, Width]


# Turn image into patches
If we set patch size = 8, then 32 x 32 image will become (32/8)^2 = 16 patches

In [4]:
patch_size = 8
patches = rearrange(x, 'b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=patch_size, p2=patch_size)

print(f"Shape after slicing: {patches.shape} -> [Batch, Num_Patches, Patch_Dim]")

Shape after slicing: torch.Size([2, 16, 192]) -> [Batch, Num_Patches, Patch_Dim]


# Patch Embedding

In [5]:
d_model = 256
patch_dim = channels * patch_size * patch_size

patch_proj = nn.Linear(patch_dim, d_model)
embedded_patches = patch_proj(patches)

print(f"Embedding Shape: {embedded_patches.shape} -> [Batch, Num_Patches, d_model]")

Embedding Shape: torch.Size([2, 16, 256]) -> [Batch, Num_Patches, d_model]


# CLS Token and Positional Embedding

In [6]:
num_patches = (image_size // patch_size) ** 2  # 16

cls_token = nn.Parameter(torch.randn(1, 1, d_model))
cls_tokens = repeat(cls_token, '1 1 d -> b 1 d', b=batch_size)
print(f"CLS Tokens Shape: {cls_tokens.shape}")

CLS Tokens Shape: torch.Size([2, 1, 256])


In [7]:
tokens = torch.cat((cls_tokens, embedded_patches), dim=1)
print(f"Sequence shape after concat: {tokens.shape} -> [Batch, 1 + Num_Patches, d_model]") # [2, 17, 256]

Sequence shape after concat: torch.Size([2, 17, 256]) -> [Batch, 1 + Num_Patches, d_model]


In [8]:
pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, d_model))
tokens = tokens + pos_embedding
print(f"Shape after adding PE: {tokens.shape}")

Shape after adding PE: torch.Size([2, 17, 256])


# Getting through the transformer block

In [13]:
from blocks import EncoderTransformerBlock

num_head = 8
d_ffn = 512
d_k = d_model // num_head
d_v = d_model // num_head

transformer_layer = EncoderTransformerBlock(
    d_model=d_model,
    num_head=num_head,
    d_ffn=d_ffn,
    d_k=d_k,
    d_v=d_v
)

encoded_tokens = transformer_layer(tokens)

print(f"Transformer block output shape: {encoded_tokens.shape}")

Transformer block output shape: torch.Size([2, 17, 256])


In [14]:
num_classes = 10

cls_output = encoded_tokens[:, 0]

mlp_head = nn.Sequential(
    nn.LayerNorm(d_model),
    nn.Linear(d_model, num_classes)
)

logits = mlp_head(cls_output)
logits

tensor([[ 0.0307, -0.1715, -1.2322, -0.8720,  0.5067, -0.3630,  0.0870,  0.2303,
          0.2772,  0.7051],
        [-0.0326,  0.0874, -1.3272, -0.7883,  0.4666, -0.2099, -0.0605,  0.3020,
          0.1832,  0.6822]], grad_fn=<AddmmBackward0>)